In [ ]:
import subprocess
import json
import os
from hipporag import HippoRAG


 # Decrypt and load environment variables from dotenvx
result = subprocess.run(['dotenvx', 'get', '--format', 'json'], capture_output=True,
text=True)

if result.returncode == 0:
    env_vars = json.loads(result.stdout)
    for key, value in env_vars.items():
        os.environ[key] = value
    print("✅ dotenvx variables loaded successfully!")
else:
    print(f"❌ Error loading dotenvx variables: {result.stderr}")

In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import openai
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from src.config import Config
from src.clients import get_clients
from src.utils.documents import process_pptx_file
from src.utils.benchmark import generate_prompt, initialize_dataframe
from src.rag_adapters import VectorRAGAdapter, LightRAGAdapter, HippoRAGAdapter, SpannerGraphRAGAdapter, AgenticRAGAdapter
from src.evaluation import add_eval_dataset, evaluate_retrieval
from src.utils.lightrag_support import initialize_lightrag

# Initialize graphRAG and DataFrame

In [ ]:
Config.setup_env()
Config.setup_directories()

clients = get_clients()

hipporag = HippoRAG(
    # llm_model_name="gpt-4o-mini-2024-07-18",
    # embedding_model_name="text-embedding-3-small"
    llm_model_name=Config.LLM_MODEL_BASE,
    llm_base_url=Config.LLM_BINDING_HOST,
    embedding_model_name=Config.EMBEDDING_MODEL,
    embedding_base_url=Config.EMBEDDING_BINDING_HOST
)

lightrag = await initialize_lightrag()





directory_path = "./documents_to_index"
file_paths = [
    os.path.join(directory_path, f)
    for f in os.listdir(directory_path)
    if os.path.isfile(os.path.join(directory_path, f))
]

adapters = {
    "vector_rag": VectorRAGAdapter(clients.qdrant_client, clients.embedding_service, clients.gemini_client),
    "lightrag": LightRAGAdapter(lightrag),
    "hipporag": HippoRAGAdapter(hipporag),
    "spanner_graph": SpannerGraphRAGAdapter(
        clients.graph_store,
        clients.embedding_service,
        clients.llm_transformer,
        Config.GRAPH_NAME,
        clients.gemini_client
    ),
    "agentic_rag": AgenticRAGAdapter(clients.qdrant_client, clients.embedding_service, clients.gemini_client)
}

df_benchmark_data = initialize_dataframe("./templates/CrossDoc_RAG_Benchmark.csv", list(adapters.keys()))
contents = process_pptx_file(file_paths)


# Generate vector and graph RAG indexes


In [ ]:
indexing_costs = {name: 0.0 for name in adapters.keys()}

for name, adapter in adapters.items():
   cost = await adapter.index(contents)
   indexing_costs[name] = cost

indexing_costs["agentic_rag"] = indexing_costs["vector_rag"]

indexing_costs_dataframe = pd.DataFrame(indexing_costs, index=["indexing_cost"])

# Retrieve contexts

In [ ]:
retrieval_intervals = {name: [] for name in adapters.keys()}
retrieval_costs = {name: [] for name in adapters.keys()}

for i in range(df_benchmark_data.shape[0]):
    query = str(df_benchmark_data.at[i, "query"])
    await asyncio.sleep(1)

    for name, adapter in adapters.items():
        start_time = datetime.datetime.now()
        context, costs = await adapter.retrieve(query)
        retrieval_intervals[name].append((datetime.datetime.now() - start_time).total_seconds())
        retrieval_costs[name].append(costs) 
        df_benchmark_data.at[i, f"results_{name}"] = context
        


retrieval_interval_dataframe = pd.DataFrame(
    retrieval_intervals,
    index=list(df_benchmark_data["query_id"])
)


retrieval_cost_dataframe = pd.DataFrame(
    retrieval_costs,
    index=list(df_benchmark_data["query_id"])
)
    

# Print retrieval results

In [ ]:
for i in range(df_benchmark_data.shape[0]):
    print(f"\nQuery: {df_benchmark_data.at[i, 'query']}\n")
    for name in adapters.keys():
        print(f"results from {name}: {df_benchmark_data.at[i, f'results_{name}']}\n\n")

# Generate LLM responses

In [ ]:
generation_intervals = {name: [] for name in adapters.keys()}
generation_costs = {name: [] for name in adapters.keys()}


async_gemini_client = clients.async_gemini_client
async_openai_client = clients.async_openai_client

@retry(wait=wait_exponential(min=4, max=60), stop=stop_after_attempt(10), retry=retry_if_exception_type(openai.RateLimitError))
async def safe_completion(prompt: str):
    return await async_gemini_client.chat.completions.with_raw_response.create(
        model=Config.LLM_MODEL_BASE,
        messages=[{"role":"user", "content":prompt}]
    )


async def timed_completion(prompt: str):
    start = datetime.datetime.now()
    r = await safe_completion(prompt)
    time = (datetime.datetime.now() - start).total_seconds()
    cost = float(r.headers.get("x-litellm-response-cost") or 0.0)
    answer = r.parse().choices[0].message.content

    return answer, time, cost

for i in range(df_benchmark_data.shape[0]):
    query = str(df_benchmark_data.at[i, "query"])

    tasks = []
    names = []
    await asyncio.sleep(1)
    for name in adapters.keys():
        if name == "agentic_rag":
            df_benchmark_data.at[i, f"actual_responses_{name}"] = adapters[name].get_response(query)
            generation_intervals[name].append(0.0)
            generation_costs[name].append(0.0)
            continue
        context = df_benchmark_data.at[i, f"results_{name}"]
        prompt = generate_prompt(query, context) # type: ignore
        names.append(name)
        tasks.append(timed_completion(prompt))
        
    responses = await asyncio.gather(*tasks)
    for name, (answer, time, cost) in zip(names, responses):
        df_benchmark_data.at[i, f"actual_responses_{name}"] = answer
        generation_intervals[name].append(time)
        generation_costs[name].append(cost)



generation_interval_dataframe = pd.DataFrame(generation_intervals, index=list(df_benchmark_data["query_id"]))
generation_cost_dataframe = pd.DataFrame(generation_costs, index=list(df_benchmark_data["query_id"]))

        

# Add dataset objects to DataFrame for eval

In [ ]:
e2e_retrieval_df = retrieval_interval_dataframe + generation_interval_dataframe
e2e_cost_df = retrieval_cost_dataframe + generation_cost_dataframe

strategies = list(adapters.keys())
add_eval_dataset(df_benchmark_data, strategies)

# Perform retrieval evaluation

In [ ]:
e2e_retrieval_df, e2e_cost_df = await evaluate_retrieval(df_benchmark_data, e2e_retrieval_df, e2e_cost_df, async_gemini_client, async_openai_client, strategies)

# Display scoring summary

In [ ]:
print("AVERAGE SCORE ACROSS ALL QUERIES")
scoring_summary_df = pd.read_csv("./results/scoring_summary.csv")
scores = scoring_summary_df.loc[scoring_summary_df["query_type"] == "ALL", "avg_score"].to_list()
methods = scoring_summary_df.loc[scoring_summary_df["query_type"] == "ALL", "system"].to_list()
sorted_scores = sorted(zip(methods, scores), key=lambda tup: tup[1])
vector_score = [sorted_scores[i][1] for i in range(len(sorted_scores)) if sorted_scores[i][0] == "vector_rag"][0]
for method, score in sorted_scores:
    delta = ((score - vector_score)/vector_score) * 100
    quotient = score/vector_score
    negative = delta < 0
    if negative:
        print(f"\n\n {method}: {int(delta)}%   |   {quotient:.2f}x")
    else:
        print(f"\n\n {method}: +{int(delta)}%   |   {quotient:.2f}x")

In [ ]:
print("GRAPH RAG AVERAGE SCORE ACROSS ALL QUERIES")
scorer_df = pd.read_csv("./results/scores.csv")
scores = [scorer_df[name].mean() for name in ["lightrag_final_score", "hipporag_final_score", "graph_final_score"]]
sorted_scores = sorted(zip(["lightrag", "hipporag", "spanner_graph"], scores), key=lambda tup: tup[1])
lowest_score = sorted_scores[0][1]
for method, score in sorted_scores:
    delta = ((score - lowest_score)/lowest_score) * 100
    quotient = score / lowest_score
    print(f"\n\n {method}: +{int(delta)}%   |    {quotient:.2f}x")

# Create latency/cost DataFrames

In [ ]:

interval_data = e2e_retrieval_df.T
cost_data = e2e_cost_df.T
indexing_data = indexing_costs_dataframe.T

# Display retrieval latency/cost metrics

In [ ]:
print("RETRIEVAL+GENERATION LATENCY PER QUERY")
interval_data_sorted = interval_data.sort_values(interval_data.columns[0])
interval_data_list = interval_data_sorted.iloc[:, 0].to_list()
methods = interval_data_sorted.index.to_list()
latency_vector = interval_data_list[0]
for method, latency in zip(methods, interval_data_list):
    delta = ((latency - latency_vector)/latency_vector)*100
    quotient = latency/latency_vector
    print(f"\n\n {method}: {int(latency)/1000} s  ---  +{int(delta)}%    |   {int(quotient)}x")


In [ ]:
print("RETRIEVAL+GENERATION COST PER QUERY")
cost_data_sorted = cost_data.sort_values(cost_data.columns[0])
cost_data_list = cost_data_sorted.iloc[:, 0].to_list()
methods = cost_data_sorted.index.to_list()
cost_vector = cost_data_list[0]
for method, cost in zip(methods, cost_data_list):
    delta = ((cost - cost_vector)/cost_vector)*100
    quotient = cost/cost_vector
    print(f"\n\n {method}: {cost:.6f} usd  ---  +{int(delta)}%    |   {int(quotient)}x")


# Display indexing metrics

In [ ]:
print("TOTAL INDEXING COST")
indexing_data_sorted = indexing_data.sort_values(indexing_data.columns[0])
indexing_data_list = indexing_data_sorted.iloc[:, 0].to_list()
methods = indexing_data_sorted.index.to_list()
cost_vector = indexing_data_list[0]
for method, cost in zip(methods, indexing_data_list):
    delta = ((cost - cost_vector)/cost_vector) * 100
    quotient = cost/cost_vector
    print(f"\n\n {method}: {cost:.6f} usd  ---  +{int(delta)}%   |   {int(quotient)}x")

# Generate plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

sns.barplot(ax=axes[0], x=interval_data_sorted.index, y=interval_data_sorted["retrieval_time"])
axes[0].set_title("Retrieval Interval")
axes[0].set(xlabel="RAG solutions", ylabel="miliseconds")

sns.barplot(ax=axes[1], x=cost_data_sorted.index, y=cost_data_sorted["retrieval_cost"])
axes[1].set_title("Retrieval Cost")
axes[1].set(xlabel="RAG solutions", ylabel="USD")

# Indexing cost per RAG method: transpose so methods become the x-axis.
sns.barplot(ax=axes[2], x=indexing_data_sorted.index, y=indexing_data_sorted["indexing_cost"])
axes[2].set_title("Indexing Cost")
axes[2].set(xlabel="RAG solutions", ylabel="USD")

plt.tight_layout()
plt.show()